[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beykyle/jitr/blob/main/examples/notebooks/integration.ipynb)

In [1]:
# Google Colab setup: installs jitr into the Colab runtime.
# Does nothing when the notebook is run anywhere else.
import sys

if "google.colab" in sys.modules:
    import subprocess

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "jitr"],
        check=True,
    )

# Numerical integration building blocks

This notebook introduces the low-level integration routines in `jitr.rmatrix`. They work on the same Lagrange-Legendre mesh that the R-matrix solver uses, so they are the natural tools for anything you evaluate on a workspace grid `ws.r`: normalizations, volume integrals, overlaps, and Fourier-Bessel transforms.

In [2]:
import numpy as np
from scipy import integrate

from jitr import rmatrix

In [3]:
def V(x, a, b):
    return a + b * x**2


def Vnl_sym(x, y, a, b):
    return a * x * y + b * (x - y) ** 2 / (y**2 + x**2)


def Vnl_asym(x, y, a, b):
    return a * x * y + b * x**2 / (y**2 + 2)

## Gauss quadrature on the mesh

`rmatrix.mesh(n, a)` builds an `n`-point mesh on $[0, a]$ (in fm). Its points `mesh.r` are the Gauss-Legendre nodes scaled to $[0, a]$, and `mesh.w` are the corresponding weights on $[0, 1]$. `rmatrix.integrate` computes $\int_0^a f(r)\, dr$ from the values $f(r_n)$:

In [4]:
a = np.pi
mesh = rmatrix.mesh(30, a)

In [5]:
analytic = 2 * a + a**3
quadrature = rmatrix.integrate(mesh, V(mesh.r, 2, 3))
print(f"quadrature = {quadrature:.6f}, analytic = {analytic:.6f}")
np.testing.assert_almost_equal(quadrature, analytic)

quadrature = 37.289462, analytic = 37.289462


`rmatrix.integrate2` computes the double integral $\int_0^a \int_0^a f(r, r')\, dr\, dr'$ of a function of two radii, such as a nonlocal potential, from its values on the mesh grid `np.meshgrid(mesh.r, mesh.r, indexing="ij")`. The integrand need not be symmetric:

In [6]:
analytic = 18.298993686404124
rn, rm = np.meshgrid(mesh.r, mesh.r, indexing="ij")
quadrature = rmatrix.integrate2(mesh, Vnl_asym(rn, rm, 1.0 / 16, 2))
print(f"quadrature = {quadrature:.6f}, analytic = {analytic:.6f}")
np.testing.assert_almost_equal(quadrature, analytic)

quadrature = 18.298994, analytic = 18.298994


The symmetric kernel `Vnl_sym` is not smooth at $r = r' = 0$, so Gauss quadrature converges more slowly there. Compare with adaptive quadrature from `scipy`:

In [7]:
analytic, _ = integrate.dblquad(
    lambda y, x: Vnl_sym(x, y, 1.0 / 16, 2), 0, 2 * np.pi, 0, 2 * np.pi
)
for n in (30, 60, 120):
    mesh2 = rmatrix.mesh(n, 2 * np.pi)
    rn, rm = np.meshgrid(mesh2.r, mesh2.r, indexing="ij")
    quadrature = rmatrix.integrate2(mesh2, Vnl_sym(rn, rm, 1.0 / 16, 2))
    print(f"n = {n:3d}: quadrature = {quadrature:.8f}, scipy = {analytic:.8f}")
np.testing.assert_almost_equal(quadrature, analytic, decimal=5)

n =  30: quadrature = 48.58016816, scipy = 48.58040026
n =  60: quadrature = 48.58038523, scipy = 48.58040026


n = 120: quadrature = 48.58039930, scipy = 48.58040026


## Fourier-Bessel transforms

`rmatrix.fourier_bessel(mesh, l, f, k)` computes $\int_0^a j_l(kr)\, r^2 f(r)\, dr$ at each $k$, and `rmatrix.fourier_bessel2(mesh, l, f, k)` the double transform $\frac{2}{\pi} \int\int j_l(kr)\, r^2 f(r, r')\, r'^2 j_l(k'r')\, dr\, dr'$ of a nonlocal function, on the grid $k \times k'$. For a Gaussian the $l = 0$ transform is known analytically:

$$
\int_0^\infty j_0(kr)\, r^2 e^{-\alpha r^2} dr = \frac{\sqrt{\pi}}{4 \alpha^{3/2}} e^{-k^2/4\alpha}.
$$

In [8]:
alpha = 0.5  # fm^-2
mesh = rmatrix.mesh(40, 15.0)  # the Gaussian is negligible beyond 15 fm
k = np.linspace(0, 4, 9)  # fm^-1

g = np.exp(-alpha * mesh.r**2)
analytic = np.sqrt(np.pi) / (4 * alpha**1.5) * np.exp(-(k**2) / (4 * alpha))
quadrature = rmatrix.fourier_bessel(mesh, 0, g, k)
print(np.abs(quadrature - analytic).max())
np.testing.assert_allclose(quadrature, analytic, atol=1e-10)

2.6583780749628305e-13


A separable nonlocal function $g(r) g(r')$ transforms into the outer product of the single transforms:

In [9]:
quadrature = rmatrix.fourier_bessel2(mesh, 0, np.outer(g, g), k)
np.testing.assert_allclose(
    quadrature, 2 / np.pi * np.outer(analytic, analytic), atol=1e-10
)

## A Lagrange-Laguerre mesh for bound states

Bound states live on $[0, \infty)$, which suits a Lagrange-Laguerre mesh. `rmatrix.laguerre(n, h)` returns its nodes $r_n = h x_n$ (with a scale $h$ in fm), the Gauss-Laguerre weights, the kinetic matrix of $-d^2/dr^2$ and the overlap matrix of the basis. In the Lagrange-mesh approximation a local potential is diagonal, $V(r_n)$, and the basis is treated as orthonormal, so the bound states are the eigenvalues of `F + np.diag(V(r))`.

For the 3D harmonic oscillator, $-u'' + r^2 u = \epsilon u$ has $\epsilon = 4n + 3$ for $l = 0$:

In [10]:
r, _, F, _ = rmatrix.laguerre(60, 0.1)
eigenvalues = np.sort(np.linalg.eigvalsh(F + np.diag(r**2)))
print(eigenvalues[:5])
np.testing.assert_allclose(eigenvalues[:5], 4 * np.arange(5) + 3, rtol=1e-6)

[ 3.  7. 11. 15. 19.]


For the Coulomb problem, $-u'' - 2u/r = \epsilon u$ has $\epsilon = -1/n^2$. The scale $h$ must be large enough for the mesh to reach the extent of the states you want:

In [11]:
for h in (0.1, 0.3, 0.5):
    r, _, F, _ = rmatrix.laguerre(40, h)
    eigenvalues = np.sort(np.linalg.eigvalsh(F - np.diag(2 / r)))
    print(f"h = {h}: {eigenvalues[:4]}")
print(f"exact:   {-1 / np.arange(1, 5) ** 2}")
np.testing.assert_allclose(eigenvalues[:4], -1 / np.arange(1, 5) ** 2, rtol=1e-6)

h = 0.1: [-1.         -0.24963259 -0.07436907  0.13780219]
h = 0.3: [-1.         -0.25       -0.11111109 -0.06238952]
h = 0.5: [-1.         -0.25       -0.11111111 -0.0625    ]
exact:   [-1.         -0.25       -0.11111111 -0.0625    ]
